In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/telco_clean.csv")

print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())

Loaded: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [2]:
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]

df["num_services"] = (df[addon_cols] == "Yes").sum(axis=1)

df["has_security_or_support"] = (
    (df["OnlineSecurity"] == "Yes") | (df["TechSupport"] == "Yes")
).astype(int)

df["auto_payment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

df["tenure_group"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, 72],
                            labels=["0-12", "13-24", "25-48", "49-72"])

for col in ["num_services", "has_security_or_support", "auto_payment", "tenure_group"]:
    print((df.groupby(col, observed=True)["Churn"].mean() * 100).round(1))
    print()

num_services
0    21.4
1    45.8
2    35.8
3    27.4
4    22.3
5    12.4
6     5.3
Name: Churn, dtype: float64

has_security_or_support
0    33.4
1    17.1
Name: Churn, dtype: float64

auto_payment
0    34.7
1    16.0
Name: Churn, dtype: float64

tenure_group
0-12     47.4
13-24    28.7
25-48    20.4
49-72     9.5
Name: Churn, dtype: float64



In [3]:
drop_cols = ["customerID", "gender", "PhoneService"]
df_model = df.drop(columns=drop_cols)

print("Before:", df.shape)
print("After: ", df_model.shape)
print()
print(df_model.dtypes)

df_model.to_csv("../data/telco_features.csv", index=False)
print()
print("Saved telco_features.csv")

Before: (7043, 25)
After:  (7043, 22)

SeniorCitizen                 int64
Partner                         str
Dependents                      str
tenure                        int64
MultipleLines                   str
InternetService                 str
OnlineSecurity                  str
OnlineBackup                    str
DeviceProtection                str
TechSupport                     str
StreamingTV                     str
StreamingMovies                 str
Contract                        str
PaperlessBilling                str
PaymentMethod                   str
MonthlyCharges              float64
TotalCharges                float64
Churn                         int64
num_services                  int64
has_security_or_support       int64
auto_payment                  int64
tenure_group               category
dtype: object

Saved telco_features.csv


In [4]:
cat_cols = ["Partner", "Dependents", "MultipleLines", "InternetService",
            "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
            "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling",
            "PaymentMethod", "tenure_group"]

num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges",
            "num_services", "has_security_or_support", "auto_payment"]

print("Text columns to encode:", len(cat_cols))
print("Number columns:", len(num_cols))
print()

for col in cat_cols:
    print(col, "->", df_model[col].nunique(), "options")

print()
print("Demo: one-hot encoding of Contract (first 5 customers)")
print(pd.get_dummies(df_model[["Contract"]].head(5)))

Text columns to encode: 14
Number columns: 7

Partner -> 2 options
Dependents -> 2 options
MultipleLines -> 3 options
InternetService -> 3 options
OnlineSecurity -> 3 options
OnlineBackup -> 3 options
DeviceProtection -> 3 options
TechSupport -> 3 options
StreamingTV -> 3 options
StreamingMovies -> 3 options
Contract -> 3 options
PaperlessBilling -> 2 options
PaymentMethod -> 4 options
tenure_group -> 4 options

Demo: one-hot encoding of Contract (first 5 customers)
   Contract_Month-to-month  Contract_One year
0                     True              False
1                    False               True
2                     True              False
3                    False               True
4                     True              False
